In [1]:
import os
import json
import requests
import warnings
from PIL import Image, ExifTags, UnidentifiedImageError, ImageFile
import time
import random
from collections import Counter

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ExifTags
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder
from IPython.display import display
from IPython.display import Image as IPyImage

# Constantes / chemins
HEADERS = {
    "User-Agent": (
        "ImageRecoProjectBot/1.0 "
        "(https://exemple-projet-univ.fr; contact: prenom.nom@etu-votre-ecole.fr) "
        "requests/2.31.0"
    ),
    "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
    "Accept-Language": "fr-FR,fr;q=0.9,en-US;q=0.8,en;q=0.7",
}

BASE_DIR = "."
IMAGES_DIR = os.path.join(BASE_DIR, "images")
DATA_DIR = os.path.join(BASE_DIR, "data")

METADATA_PATH = os.path.join(DATA_DIR, "images_metadata.json")
LABELS_PATH = os.path.join(DATA_DIR, "images_labels.json")
USERS_PATH = os.path.join(DATA_DIR, "users.json")

os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

url_api = "https://commons.wikimedia.org/w/api.php?action=query&generator=search&gsrsearch=cat&gsrnamespace=6&gsrlimit=50&prop=imageinfo&iiprop=url|size|mime|extmetadata&format=json"

COMMONS_API_URL = "https://commons.wikimedia.org/w/api.php"

PEXELS_API_KEY = "ffSblWpnhOlWSqdXuTQL3rEK7eOLfPj6TV4R5WItWPpArJhfIaU9kmac"

In [2]:
def search_pexels_images(query, api_key, limit=50):
    """
    Cherche des images sur Pexels
    """
    url = "https://api.pexels.com/v1/search"

    headers = {
        "Authorization": api_key
    }

    params = {
        "query": query,
        "per_page": limit
    }

    resp = requests.get(url, headers=headers, params=params, timeout=15)
    resp.raise_for_status()
    data = resp.json()

    images = []

    for photo in data.get("photos", []):
        images.append({
            "title": photo.get("alt"),
            "url": photo["src"].get("original"),
            "width": photo.get("width"),
            "height": photo.get("height"),
            "mime": "image/jpeg",  # Pexels → JPEG majoritairement
            "license_short_name": "Pexels License",
            "license_url": "https://www.pexels.com/license/",
            "artist": photo.get("photographer"),
            "credit": photo.get("photographer_url"),
            "source": "Pexels"
        })

    return images

def search_commons_images(query, limit=50):
    """
    Cherche des images sur Wikimedia Commons et renvoie une liste
    de dicts avec url + licence + auteur, etc.
    """
    params = {
        "action": "query",
        "generator": "search",
        "gsrsearch": query,
        "gsrnamespace": 6,        # namespace 'File:'
        "gsrlimit": limit,
        "prop": "imageinfo",
        "iiprop": "url|size|mime|extmetadata",
        "format": "json",
        "origin": "*",
    }

    resp = requests.get(COMMONS_API_URL, params=params, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    data = resp.json()

    if "query" not in data or "pages" not in data["query"]:
        return []

    images = []
    for page in data["query"]["pages"].values():
        title = page.get("title")
        imageinfo = page.get("imageinfo", [])
        if not imageinfo:
            continue
        info = imageinfo[0]
        url = info.get("url")
        width = info.get("width")
        height = info.get("height")
        mime = info.get("mime")
        extmeta = info.get("extmetadata", {})

        license_short_name = extmeta.get("LicenseShortName", {}).get("value")
        license_url = extmeta.get("LicenseUrl", {}).get("value")
        artist = extmeta.get("Artist", {}).get("value")
        credit = extmeta.get("Credit", {}).get("value")

        images.append({
            "title": title,
            "url": url,
            "width": width,
            "height": height,
            "mime": mime,
            "license_short_name": license_short_name,
            "license_url": license_url,
            "artist": artist,
            "credit": credit,
            "source": "Wikimedia Commons",
        })

    return images

def search_all_images(query, limit=50):
    Wikimedia_images = search_commons_images(query, limit)
    pexels_images = search_pexels_images(query, PEXELS_API_KEY, limit)

    return Wikimedia_images + pexels_images

In [3]:
all_sources = []
topics = [
    ("cat", 40),
    ("dog", 40),
    ("mountain landscape", 40),
]

for query, limit in topics:
    results = search_all_images(query, limit=limit)
    all_sources.extend(results)

print(f"{len(all_sources)} images trouvées avant dédoublonnage")

240 images trouvées avant dédoublonnage


In [4]:
# Facultatif : évite certains problèmes sur images tronquées
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Extensions acceptées comme images
VALID_IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff", ".webp"}


import os
import re

def normalize_base_name(name):

    m = re.fullmatch(r"(img_)(\d+)", name.lower())
    if not m:
        return name.lower()

    prefix, number = m.groups()
    return f"{prefix}{int(number):03d}"


def get_used_base_names(folder):

    used = set()

    for f in os.listdir(folder):
        base, _ = os.path.splitext(f)
        used.add(normalize_base_name(base))

    return used


def find_next_index(used_base_names, start_idx=1):

    idx = start_idx

    while True:
        candidate = normalize_base_name(f"img_{idx:03d}")
        if candidate not in used_base_names:
            used_base_names.add(candidate)
            return idx
        idx += 1


def download_image(url, dest_folder, idx):
    response = requests.get(url, headers=HEADERS, timeout=10)
    response.raise_for_status()

    # Détection extension
    ext = os.path.splitext(url.split("?")[0])[1].lower()
    if not ext:
        ext = ".jpg"

    # 🔥 NOUVEAU : gestion unicité globale
    used_base_names = get_used_base_names(dest_folder)
    idx = find_next_index(used_base_names, idx)

    filename = f"img_{idx:03d}{ext}"
    filepath = os.path.join(dest_folder, filename)

    with open(filepath, "wb") as f:
        f.write(response.content)

    return filename, filepath, ext
def extract_exif(filepath):
    try:
        with warnings.catch_warnings():
            warnings.filterwarnings(
                "ignore",
                message=r"Corrupt EXIF data.*",
                category=UserWarning
            )
            with Image.open(filepath) as img:
                raw_exif = img.getexif()

        exif_data = {}
        for tag_id, value in raw_exif.items():
            tag = ExifTags.TAGS.get(tag_id, tag_id)
            exif_data[tag] = str(value)

        return exif_data

    except Exception:
        return {}

In [ ]:
# Recherche d'images par thèmes
topics = [
    ("cat", 40),
    ("dog", 40),
    ("mountain landscape", 40),
]

all_sources = []
for query, limit in topics:
    results = search_all_images(query, limit=limit)
    all_sources.extend(results)

print(f"{len(all_sources)} images trouvées avant dédoublonnage")

metadata_list = []

for idx, src in enumerate(all_sources, start=1):
    url = src.get("url")
    license_info = src.get("license", "unknown")
    source_site = src.get("source", "unknown")

    try:
        filename, filepath, ext = download_image(url, IMAGES_DIR, idx)
        time.sleep(0.75)

        # Ignorer les fichiers qui ne sont pas des images
        if ext not in VALID_IMAGE_EXTENSIONS:
            print(f"Image {idx} ignorée ({url}) : extension non supportée ({ext})")
            continue

        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                with Image.open(filepath) as img:
                    width, height = img.size
                    img_format = img.format

        except UnidentifiedImageError:
            print(f"Erreur sur l’image {idx} ({url}) : fichier non reconnu comme image")
            continue

        except Image.DecompressionBombError:
            print(f"Erreur sur l’image {idx} ({url}) : image trop grande, ignorée")
            continue

        file_size_bytes = os.path.getsize(filepath)
        file_size_kb = round(file_size_bytes / 1024, 2)

        exif = extract_exif(filepath)

        metadata = {
            "filename": filename,
            "width": width,
            "height": height,
            "format": img_format,
            "file_size_kb": file_size_kb,
            "url": url,
            "license": license_info,
            "source": source_site,
            "exif": exif,
        }
        metadata_list.append(metadata)

    except Exception as e:
        print(f"Erreur sur l’image {idx} ({url}): {e}")

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata_list, f, ensure_ascii=False, indent=2)

print(f"{len(metadata_list)} images traitées et sauvegardées dans {METADATA_PATH}")

240 images trouvées avant dédoublonnage


In [ ]:
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata_list = json.load(f)

def compute_orientation(width, height):
    if width > height:
        return "paysage"
    elif height > width:
        return "portrait"
    else:
        return "carré"

def compute_size_category(max_dim):
    if max_dim < 500:
        return "vignette"
    elif max_dim <= 1500:
        return "moyenne"
    else:
        return "grande"

def get_dominant_colors(image_path, n_colors=4):
    img = Image.open(image_path).convert("RGB")
    img_small = img.resize((150, 150))
    pixels = np.array(img_small).reshape(-1, 3)

    kmeans = KMeans(n_clusters=n_colors, n_init=5, random_state=42)
    kmeans.fit(pixels)
    centers = kmeans.cluster_centers_.astype(int)

    return [center.tolist() for center in centers]

def rgb_to_basic_name(rgb):
    r, g, b = rgb
    if max(rgb) < 80:
        return "noir"
    if r > g and r > b:
        return "rouge"
    if g > r and g > b:
        return "vert"
    if b > r and b > g:
        return "bleu"
    return "gris"

In [ ]:
import os
import json
import re
from urllib.parse import urlparse, unquote

import spacy

# Charge un modèle spaCy
# À installer avant si besoin :
# python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm")


def clean_url_for_ner(url):
    """
    Garde uniquement la partie après le nom de domaine,
    décode l'URL et remplace les séparateurs par des espaces.
    """
    if not url:
        return ""

    parsed = urlparse(url)

    # On garde path + query éventuelle
    raw_text = f"{parsed.path} {parsed.query}".strip()

    # Décoder les caractères encodés (%20, etc.)
    raw_text = unquote(raw_text)

    # Enlever l'extension finale éventuelle
    raw_text = re.sub(r"\.(jpg|jpeg|png|webp|gif|bmp|tiff)\b", " ", raw_text, flags=re.IGNORECASE)

    # Remplacer séparateurs courants par des espaces
    raw_text = re.sub(r"[/_\-+=&?.,:;()%#@!~\[\]{}|\\]+", " ", raw_text)

    # Supprimer les suites de chiffres trop isolées
    raw_text = re.sub(r"\b\d+\b", " ", raw_text)

    # Normaliser les espaces
    raw_text = re.sub(r"\s+", " ", raw_text).strip()

    return raw_text


def extract_tags_from_url(url):
    """
    Extrait des tags à partir de l'URL nettoyée en utilisant spaCy.
    - priorité aux entités nommées
    - fallback sur les tokens significatifs si aucune entité n'est trouvée
    """
    cleaned_text = clean_url_for_ner(url)

    if not cleaned_text:
        return []

    doc = nlp(cleaned_text)
    tags = []

    # 1) Entités nommées détectées par spaCy
    allowed_entity_labels = {
        "PERSON", "NORP", "FAC", "ORG", "GPE", "LOC",
        "PRODUCT", "EVENT", "WORK_OF_ART", "LANGUAGE"
    }

    for ent in doc.ents:
        if ent.label_ in allowed_entity_labels:
            tag = ent.text.strip().lower()
            if len(tag) > 1:
                tags.append(tag)

    # 2) Fallback : tokens significatifs si spaCy ne trouve rien
    if not tags:
        for token in doc:
            if (
                not token.is_stop
                and not token.is_punct
                and not token.is_space
                and not token.like_num
                and len(token.text) > 2
                and token.pos_ in {"NOUN", "PROPN", "ADJ"}
            ):
                tags.append(token.lemma_.lower())

    # Déduplication en conservant l'ordre
    seen = set()
    unique_tags = []
    for tag in tags:
        if tag not in seen:
            seen.add(tag)
            unique_tags.append(tag)

    return unique_tags


labels = {}

for meta in metadata_list:
    filename = meta["filename"]
    width = meta["width"]
    height = meta["height"]
    url = meta.get("url", "")

    orientation = compute_orientation(width, height)
    size_cat = compute_size_category(max(width, height))

    image_path = os.path.join(IMAGES_DIR, filename)

    try:
        colors = get_dominant_colors(image_path, n_colors=4)
        color_names = [rgb_to_basic_name(c) for c in colors]
    except Exception as e:
        print(f"Impossible d'extraire les couleurs pour {filename}: {e}")
        colors = []
        color_names = []

    try:
        url_tags = extract_tags_from_url(url) if url else []
    except Exception as e:
        print(f"Impossible d'extraire les tags URL pour {filename}: {e}")
        url_tags = []

    labels[filename] = {
        "predominant_colors": colors,
        "color_names": color_names,
        "orientation": orientation,
        "size_category": size_cat,
        "tags": url_tags,
        "cleaned_url_text": clean_url_for_ner(url) if url else ""
    }

with open(LABELS_PATH, "w", encoding="utf-8") as f:
    json.dump(labels, f, ensure_ascii=False, indent=2)

print(f"Annotations sauvegardées dans {LABELS_PATH}")

In [ ]:
with open(LABELS_PATH, "r", encoding="utf-8") as f:
    image_labels = json.load(f)

image_filenames = list(image_labels.keys())
print(f"{len(image_filenames)} images avec labels chargées.")

def most_common_or_none(lst):
    if not lst:
        return None
    return Counter(lst).most_common(1)[0][0]

def build_user_profile(user_id, favorite_images):
    colors = []
    orientations = []
    sizes = []
    tags = []

    for fname in favorite_images:
        info = image_labels[fname]
        colors.extend(info.get("color_names", []))
        orientations.append(info.get("orientation"))
        sizes.append(info.get("size_category"))
        tags.extend(info.get("tags", []))

    profile = {
        "user_id": user_id,
        "favorite_colors": list({c for c in colors if c}),
        "favorite_orientation": most_common_or_none([o for o in orientations if o]),
        "favorite_size": most_common_or_none([s for s in sizes if s]),
        "favorite_tags": list({t for t in tags if t}),
        "favorite_images": favorite_images,
    }
    return profile

def filter_images_by_color(color):
    return [
        fname for fname, info in image_labels.items()
        if color in info.get("color_names", [])
    ]

def filter_images_by_orientation(orientation):
    return [
        fname for fname, info in image_labels.items()
        if info.get("orientation") == orientation
    ]

def filter_images_by_size(size_cat):
    return [
        fname for fname, info in image_labels.items()
        if info.get("size_category") == size_cat
    ]

users = {}
random.seed(42)

# User 1 : aime les images rouges en paysage
candidates_u1 = list(
    set(filter_images_by_color("rouge"))
    & set(filter_images_by_orientation("paysage"))
)
favorites_u1 = random.sample(
    candidates_u1, k=min(12, len(candidates_u1))
) if candidates_u1 else random.sample(image_filenames, k=12)
users["user_001"] = build_user_profile("user_001", favorites_u1)

# User 2 : aime les images bleues en portrait
candidates_u2 = list(
    set(filter_images_by_color("bleu"))
    & set(filter_images_by_orientation("portrait"))
)
favorites_u2 = random.sample(
    candidates_u2, k=min(12, len(candidates_u2))
) if candidates_u2 else random.sample(image_filenames, k=12)
users["user_002"] = build_user_profile("user_002", favorites_u2)

# User 3 : grandes images
candidates_u3 = filter_images_by_size("grande")
favorites_u3 = random.sample(
    candidates_u3, k=min(12, len(candidates_u3))
) if candidates_u3 else random.sample(image_filenames, k=12)
users["user_003"] = build_user_profile("user_003", favorites_u3)

# User 4 : petites vignettes
candidates_u4 = filter_images_by_size("vignette")
favorites_u4 = random.sample(
    candidates_u4, k=min(12, len(candidates_u4))
) if candidates_u4 else random.sample(image_filenames, k=12)
users["user_004"] = build_user_profile("user_004", favorites_u4)

# User 5 : goûts mixtes
favorites_u5 = random.sample(
    image_filenames, k=min(12, len(image_filenames))
)
users["user_005"] = build_user_profile("user_005", favorites_u5)

with open(USERS_PATH, "w", encoding="utf-8") as f:
    json.dump(users, f, ensure_ascii=False, indent=2)

print(f"{len(users)} profils utilisateurs sauvegardés dans {USERS_PATH}")

In [ ]:
with open(LABELS_PATH, "r", encoding="utf-8") as f:
    image_labels = json.load(f)
with open(USERS_PATH, "r", encoding="utf-8") as f:
    users = json.load(f)

rows = []
for fname, info in image_labels.items():
    color_names = info.get("color_names", [])
    main_color = color_names[0] if color_names else None

    tags = info.get("tags", [])
    main_tag = tags[0] if tags else None

    rows.append({
        "filename": fname,
        "main_color": main_color,
        "orientation": info.get("orientation"),
        "size_category": info.get("size_category"),
        "main_tag": main_tag,
    })

df_images = pd.DataFrame(rows)
print(df_images.head())
print(df_images["main_color"].value_counts())

feature_cols = ["main_color", "orientation", "size_category", "main_tag"]

encoders = {}
encoded_features = []

for col in feature_cols:
    le = LabelEncoder()
    values = df_images[col].fillna("unknown")
    encoded_col = le.fit_transform(values)
    encoders[col] = le
    encoded_features.append(encoded_col)

X = np.vstack(encoded_features).T

k = 5
kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
df_images["cluster"] = kmeans.fit_predict(X)

df_images.head()

In [ ]:
all_images = list(image_labels.keys())

def get_user_top_clusters(user_id, top_k=2):
    user = users[user_id]
    favs = user["favorite_images"]

    clusters = []
    for fname in favs:
        info = image_labels.get(fname)
        if info is None:
            continue
        # chercher le cluster via df_images
        cl = df_images.loc[df_images["filename"] == fname, "cluster"]
        if not cl.empty:
            clusters.append(int(cl.values[0]))

    if not clusters:
        return []

    counts = Counter(clusters).most_common(top_k)
    return [c for c, _ in counts]


def recommend_images(user_id, n_recommendations=5):
    user = users[user_id]
    favs = set(user["favorite_images"])
    top_clusters = get_user_top_clusters(user_id, top_k=2)

    if not top_clusters:
        candidates = [img for img in all_images if img not in favs]
    else:
        candidates = []
        for fname in all_images:
            if fname in favs:
                continue
            cl = df_images.loc[df_images["filename"] == fname, "cluster"]
            if not cl.empty and int(cl.values[0]) in top_clusters:
                candidates.append(fname)

    fav_colors = set(user.get("favorite_colors") or [])

    def score_image(fname):
        info = image_labels[fname]
        colors = set(info.get("color_names") or [])
        return len(colors & fav_colors)

    candidates_sorted = sorted(candidates, key=score_image, reverse=True)

    if len(candidates_sorted) > n_recommendations:
        candidates_sorted = candidates_sorted[: n_recommendations * 3]
        random.shuffle(candidates_sorted)

    selected = candidates_sorted[:n_recommendations]

    recommendations = []
    for fname in selected:
        info = image_labels[fname]
        reason_parts = []

        colors = info.get("color_names") or []
        if fav_colors and set(colors) & fav_colors:
            reason_parts.append(
                f"couleurs proches des préférences de l'utilisateur ({', '.join(sorted(set(colors) & fav_colors))})"
            )

        ori = info.get("orientation")
        size = info.get("size_category")
        if ori:
            reason_parts.append(f"orientation {ori}")
        if size:
            reason_parts.append(f"taille {size}")

        cl = df_images.loc[df_images["filename"] == fname, "cluster"]
        if not cl.empty:
            reason_parts.append(f"cluster {int(cl.values[0])}")

        reason = " ; ".join(reason_parts) if reason_parts else "similaire à vos images favorites"
        recommendations.append((fname, reason))

    return recommendations

# Test rapide
recs = recommend_images("user_001", 5)
for fname, reason in recs:
    print(fname, "→", reason)

In [ ]:
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata_list = json.load(f)
with open(LABELS_PATH, "r", encoding="utf-8") as f:
    image_labels = json.load(f)
with open(USERS_PATH, "r", encoding="utf-8") as f:
    users = json.load(f)

df_meta = pd.DataFrame(metadata_list)
df_labels = pd.DataFrame([
    {"filename": fname, **info} for fname, info in image_labels.items()
])

# 1) Orientation
plt.figure(figsize=(5, 4))
df_labels["orientation"].value_counts().plot(kind="bar")
plt.title("Nombre d'images par orientation")
plt.xlabel("Orientation")
plt.ylabel("Nombre d'images")
plt.tight_layout()
plt.show()

# 2) Taille
plt.figure(figsize=(5, 4))
df_labels["size_category"].value_counts().plot(kind="bar")
plt.title("Nombre d'images par catégorie de taille")
plt.xlabel("Catégorie de taille")
plt.ylabel("Nombre d'images")
plt.tight_layout()
plt.show()

# 3) Formats
plt.figure(figsize=(5, 4))
df_meta["format"].value_counts().plot(kind="pie", autopct="%1.1f%%")
plt.title("Distribution des formats d'images")
plt.ylabel("")
plt.tight_layout()
plt.show()

# 4) Couleurs globales
all_colors = []
for info in image_labels.values():
    all_colors.extend(info.get("color_names", []))
df_colors = pd.Series(all_colors)

plt.figure(figsize=(5, 4))
df_colors.value_counts().plot(kind="bar")
plt.title("Fréquence des couleurs dominantes")
plt.xlabel("Couleur")
plt.ylabel("Fréquence")
plt.tight_layout()
plt.show()

# 5) Couleurs favorites par utilisateur
user_colors = []
for uid, uinfo in users.items():
    for c in uinfo.get("favorite_colors", []):
        user_colors.append({"user_id": uid, "color": c})

df_user_colors = pd.DataFrame(user_colors)
if not df_user_colors.empty:
    plt.figure(figsize=(6, 4))
    pd.crosstab(df_user_colors["user_id"], df_user_colors["color"]).plot(
        kind="bar", stacked=True
    )
    plt.title("Couleurs favorites par utilisateur")
    plt.xlabel("Utilisateur")
    plt.ylabel("Nombre de couleurs")
    plt.tight_layout()
    plt.show()

In [ ]:
def test_data_integrity():
    images_files = [
        f for f in os.listdir(IMAGES_DIR)
        if os.path.isfile(os.path.join(IMAGES_DIR, f))
    ]
    print("Nombre de fichiers dans images/ :", len(images_files))

    IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".gif", ".tif", ".tiff"}
    image_files_filtered = [
        f for f in images_files
        if os.path.splitext(f)[1].lower() in IMAGE_EXTENSIONS
    ]
    print("Nombre de fichiers image pris en compte :", len(image_files_filtered))

    assert len(image_files_filtered) >= 100, "Besoin d'au moins 100 images (hors vidéos, etc.)"

    meta_filenames = {m["filename"] for m in metadata_list}
    label_filenames = set(image_labels.keys())

    missing_meta = [f for f in image_files_filtered if f not in meta_filenames]
    missing_labels = [f for f in image_files_filtered if f not in label_filenames]

    if missing_meta:
        print("Attention : métadonnées manquantes pour :", missing_meta)
    if missing_labels:
        print("Attention : labels manquants pour :", missing_labels)

    covered = [
        f for f in image_files_filtered
        if f in meta_filenames and f in label_filenames
    ]
    coverage_ratio = len(covered) / len(image_files_filtered)
    print(f"Couverture métadonnées+labels : {coverage_ratio:.2%}")
    assert coverage_ratio > 0.9, "Moins de 90% des images ont métadonnées + labels"

    print("test_data_integrity OK (avec warnings éventuels)")


def test_colors_labels():
    for fname, info in image_labels.items():
        colors = info.get("predominant_colors", [])
        for rgb in colors:
            assert len(rgb) == 3, f"Couleur RGB invalide pour {fname}"
            r, g, b = rgb
            assert 0 <= r <= 255 and 0 <= g <= 255 and 0 <= b <= 255, \
                f"Valeurs RGB hors bornes pour {fname}"
    print("test_colors_labels OK")


def test_recommendation_system():
    user_ids = list(users.keys())
    assert user_ids, "Pas d'utilisateurs dans users.json"
    uid = user_ids[0]

    recs = recommend_images(uid, n_recommendations=5)
    assert len(recs) == 5, "La fonction doit retourner exactement 5 recommandations"

    favs = set(users[uid]["favorite_images"])
    rec_filenames = [fname for fname, _ in recs]
    assert not (favs & set(rec_filenames)), \
        "Les recommandations contiennent des images déjà favorites"

    print("test_recommendation_system OK pour", uid)


test_colors_labels()
test_recommendation_system()
test_data_integrity()

In [ ]:
def show_user_recommendations(user_id, n_recommendations=5):
    print(f"Utilisateur : {user_id}")
    favs = users[user_id]["favorite_images"]
    print("\nImages favorites :")
    for fname in favs[:5]:
        print("-", fname)
        display(IPyImage(filename=os.path.join(IMAGES_DIR, fname)))

    print("\nRecommandations :")
    recs = recommend_images(user_id, n_recommendations=n_recommendations)
    for fname, reason in recs:
        print(f"{fname} → {reason}")
        display(IPyImage(filename=os.path.join(IMAGES_DIR, fname)))

show_user_recommendations("user_001", n_recommendations=5)